# P3 — Dataset Preparation and Experimental Setup

## 3.1 Introduction and Objectives

With the source data validated in P1 and the ICD-based reference labels constructed in P2, the next step is to prepare a shared dataset for comparing different NLP methods.

Since chief complaints are the only model input in this study, I first examine their textual characteristics and apply a literature-informed preprocessing strategy before integrating them with encounter identifiers and reference labels.

The objectives of this notebook are to:

- Examine chief complaint length, repetition textual variation and other characteristics.
- Apply minimal common preprocessing while preserving potentially useful information.
- Construct and validate the encounter-level modeling dataset.
- Establish patient-grouped training, validation and test sets.
- Construct nested training subsets for the training data efficiency experiment.

All NLP methods will use the same eligible encounters, reference labels and experimental partitions. Additional text processing will be handled within each model-specific pipeline.

## 3.2 Chief Complaint Characteristics

Before applying text preprocessing, I examine the characteristics of the raw chief complaints to better understand the available textual information and potential challenges for different NLP methods.

### Literature-Informed Analysis

Previous studies have examined different aspects of clinical text when developing NLP classification methods. The following table summarizes their main considerations and how they inform the analysis in this study.

| Literature | Text Characteristics Considered | Implications for My Study |
|---|---|---|
| Lee et al. (2019) | Text length, unique tokens, medical terminology, abbreviations and misspellings | Examine text length and linguistic variation. |
| Valmianski et al. (2019) | Character length, word count, unique text entries, spelling variations and contextual complexity | Understand short-text characteristics, text diversity and potential challenges for lexical and contextual representations. |
| Chang et al. (2020) | Variation in text length and quality, multiple symptoms and semantic overlap | Recognize the limitations of short clinical texts and the ambiguity of symptom descriptions. |
| Miotto et al. (2020) | Document length, vocabulary size, low-frequency words, spelling variations and duplicate notes | Consider vocabulary sparsity and text quality, while recognizing the differences between clinical notes and short chief complaints. |
| Xie et al. (2022) | Missing values, duplicate records and data quality | Establish reliable input data and consistent experimental conditions. |
| Weidman et al. (2026) | Extremely short EMS chief complaints, word count and lexical vocabulary | Provides relevant context for examining text brevity and the potential effectiveness of lexical representations. |

### Analysis Scope

I distinguish general text characterization from model-specific text exploration. At this stage, the focus is on understanding the source text and informing common preprocessing without using ICD reference labels or developing model-specific features.

| Analysis | Current Stage | Model Development |
|---|:---:|:---:|
| Missing, empty and whitespace-only text | Yes | — |
| Character length and word count distribution | Yes | — |
| Unique complaints and text repetition | Yes | — |
| Capitalization patterns | Yes | — |
| Punctuation and numeric presence | Yes | — |
| Whitespace characteristics | Yes | — |
| Unusual text formats | Yes | — |
| Detailed vocabulary and word frequency analysis | — | As needed |
| Rare words and spelling variations | — | As needed |
| Specific abbreviation patterns | — | As needed |
| Common complaint expressions | — | As needed |
| LBP-related keyword analysis | — | As needed |
| Text patterns associated with ICD reference labels | — | As needed |
| Model-specific tokenization characteristics | — | As needed |

General text characterization is performed on the source dataset. Detailed lexical exploration, task-specific analysis and model development are restricted to the appropriate development partitions after splitting. Corpus-dependent transformations are fitted only on the current training subset.

The purpose of this analysis is to understand the text, not to classify encounters or establish additional clinical inclusion and exclusion criteria.

### Input Data and Availability

P1 established chief complaint availability and encounter-level data integrity in the source database. I first validate the exported data and examine the availability of usable text before proceeding with text characterization.

The original text is preserved during CSV import to avoid automatically interpreting strings such as `NA` or `N/A` as missing values.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_DIR = Path("../results/01_dataset_exploration")
LABEL_DIR = Path("../results/02_reference_labels")
RESULTS_DIR = Path("../results/03_experimental_setup")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load raw chief complaint data
chief_raw = pd.read_csv(
    INPUT_DIR / "chief_complaint_raw.csv",
    dtype={"chiefcomplaint": str},
    keep_default_na=False
)

# Validate input data
input_validation = pd.DataFrame({
    "Check": [
        "Expected columns",
        "Expected encounter count",
        "No missing or empty stay_id",
        "Unique stay_id"
    ],
    "Passed": [
        set(chief_raw.columns) == {"stay_id", "chiefcomplaint"},
        len(chief_raw) == 425087,
        chief_raw["stay_id"].notna().all()
        and chief_raw["stay_id"].astype(str).str.strip().ne("").all(),
        chief_raw["stay_id"].is_unique
    ]
})

display(input_validation)

assert input_validation["Passed"].all(), "Input validation failed"

# Examine text availability
raw_text = chief_raw["chiefcomplaint"]

missing_mask = raw_text.isna()
empty_mask = raw_text.notna() & raw_text.eq("")
whitespace_mask = (
    raw_text.notna()
    & raw_text.ne("")
    & raw_text.str.strip().eq("")
)

available_mask = ~(missing_mask | empty_mask | whitespace_mask)

availability_summary = pd.DataFrame({
    "Status": [
        "Missing (NaN)",
        "Empty string",
        "Whitespace only",
        "Available"
    ],
    "Encounters": [
        missing_mask.sum(),
        empty_mask.sum(),
        whitespace_mask.sum(),
        available_mask.sum()
    ]
})

availability_summary["Percentage (%)"] = (
    availability_summary["Encounters"] / len(chief_raw) * 100
)

assert availability_summary["Encounters"].sum() == len(chief_raw)

availability_summary.loc[len(availability_summary)] = [
    "Total",
    availability_summary["Encounters"].sum(),
    availability_summary["Percentage (%)"].sum()
]

display(availability_summary.round(4))

# Prepare available text for descriptive analysis
chief_available = chief_raw.loc[available_mask].copy()
text = chief_available["chiefcomplaint"]

print("Input validation: PASSED")

,Check,Passed
0,Expected columns,True
1,Expected encounter count,True
2,No missing or empty stay_id,True
3,Unique stay_id,True


,Status,Encounters,Percentage (%)
0,Missing (NaN),0,0.0000
1,Empty string,22,0.0052
2,Whitespace only,0,0.0000
3,Available,425065,99.9948
4,Total,425087,100.0000


Input validation: PASSED


**Observation:** Chief complaint availability is consistent with the source data analysis in P1. The 22 missing values identified in BigQuery are represented as empty strings after CSV import. A total of 425,065 encounters (99.9948%) contain available chief complaint text and are retained for descriptive analysis.

### Text Length Distribution

Text length is particularly relevant to this study because short complaints may provide limited context for NLP models. Valmianski et al. (2019) suggested that short text length may partly explain the strong performance of TF-IDF in their chief complaint classification task.

I examine both character counts and whitespace-separated word counts to understand the amount of text available to the models without introducing model-specific tokenization.

In [2]:
# Calculate character length and whitespace-separated word count
text_lengths = pd.DataFrame({
    "Character Count": text.str.len(),
    "Word Count": text.str.split().str.len()
})

assert len(text_lengths) == len(chief_available)
assert text_lengths["Word Count"].gt(0).all()

# Descriptive statistics
length_summary = text_lengths.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T

length_summary = length_summary.rename(columns={
    "count": "Count",
    "mean": "Mean",
    "std": "Std",
    "min": "Min",
    "25%": "P25",
    "50%": "Median",
    "75%": "P75",
    "90%": "P90",
    "95%": "P95",
    "99%": "P99",
    "max": "Max"
})

print("Text Length Summary")

length_summary["Count"] = length_summary["Count"].astype(int)
display(length_summary)

# Word count distribution
word_count_groups = pd.cut(
    text_lengths["Word Count"],
    bins=[0, 1, 3, 5, 10, 20, np.inf],
    labels=["1", "2–3", "4–5", "6–10", "11–20", "21+"]
)

word_count_summary = (
    word_count_groups.value_counts(sort=False)
    .rename_axis("Word Count")
    .reset_index(name="Encounters")
)

word_count_summary["Percentage (%)"] = (
    word_count_summary["Encounters"] / len(text_lengths) * 100
)

assert word_count_summary["Encounters"].sum() == len(text_lengths)

# Add total row
word_count_summary["Word Count"] = word_count_summary["Word Count"].astype(str)

word_count_summary.loc[len(word_count_summary)] = [
    "Total",
    word_count_summary["Encounters"].sum(),
    word_count_summary["Percentage (%)"].sum()
]

print("Word Count Distribution")
display(word_count_summary.round(2))

Text Length Summary


,Count,Mean,Std,Min,P25,Median,P75,P90,P95,P99,Max
Character Count,425065,14.556014,8.050813,1.0,8.0,13.0,19.0,26.0,30.0,39.0,136.0
Word Count,425065,2.430217,1.237054,1.0,2.0,2.0,3.0,4.0,5.0,6.0,16.0


Word Count Distribution


,Word Count,Encounters,Percentage (%)
0,1,102378,24.09
1,2–3,257091,60.48
2,4–5,55807,13.13
3,6–10,9749,2.29
4,11–20,40,0.01
5,21+,0,0.00
6,Total,425065,100.00


**Observation:** Chief complaints in this dataset are extremely short, with a mean word count of 2.43 and a median of 2. Approximately 84.57% contain no more than three words, and 24.09% consist of a single word.

The median word count is lower than the seven words reported by Valmianski et al. (2019) and matches the two words reported by Weidman et al. (2026) in a large EMS chief complaint study. Although the text sources and counting methods may differ, these findings provide relevant context for comparing lexical and contextual NLP methods on extremely short clinical texts.

The high proportion of single-word complaints also supports retaining valid short texts rather than applying minimum-length filters designed for longer clinical notes.

### Text Diversity and Repetition

In addition to text length, the diversity and repetition of chief complaints are relevant to understanding the characteristics of short clinical texts.

I examine the number of distinct complaint strings, their occurrence frequencies and the concentration of frequently repeated expressions. This analysis uses exact raw text matches without case normalization or other preprocessing.

Identical complaints from different encounters are retained because repeated expressions may reflect common clinical presentations and documentation practices rather than duplicate records.

In [3]:
# Calculate exact complaint frequencies
complaint_counts = text.value_counts()

n_encounters = len(text)
n_unique = len(complaint_counts)
n_singletons = complaint_counts.eq(1).sum()
n_repeated = complaint_counts.gt(1).sum()
n_repeated_encounters = complaint_counts[
    complaint_counts.gt(1)
].sum()

assert complaint_counts.sum() == n_encounters

# 1. Text diversity summary
diversity_summary = pd.DataFrame({
    "Metric": [
        "Total available complaints",
        "Distinct complaint strings",
        "Strings appearing once",
        "Strings appearing more than once",
        "Encounters with repeated strings",
        "Distinct-to-total ratio (%)",
        "Singleton strings (%)",
        "Encounters with repeated strings (%)"
    ],
    "Value": [
        n_encounters,
        n_unique,
        n_singletons,
        n_repeated,
        n_repeated_encounters,
        n_unique / n_encounters * 100,
        n_singletons / n_unique * 100,
        n_repeated_encounters / n_encounters * 100
    ]
})

print("Text Diversity Summary")
display(diversity_summary.round(2))

# 2. Complaint frequency distribution
frequency_groups = pd.cut(
    complaint_counts,
    bins=[0, 1, 5, 10, 100, 1000, np.inf],
    labels=["1", "2–5", "6–10", "11–100", "101–1000", "1001+"]
)

frequency_summary = pd.DataFrame({
    "Frequency Group": frequency_groups.astype(str),
    "Encounters": complaint_counts.values
})

frequency_summary = (
    frequency_summary
    .groupby("Frequency Group", sort=False)
    .agg(
        Distinct_Strings=("Encounters", "size"),
        Encounters=("Encounters", "sum")
    )
    .reindex(["1", "2–5", "6–10", "11–100", "101–1000", "1001+"], fill_value=0)
    .reset_index()
    .rename(columns={"Distinct_Strings": "Distinct Strings"})
)

frequency_summary["Distinct Strings (%)"] = (
    frequency_summary["Distinct Strings"] / n_unique * 100
)

frequency_summary["Encounters (%)"] = (
    frequency_summary["Encounters"] / n_encounters * 100
)

assert frequency_summary["Distinct Strings"].sum() == n_unique
assert frequency_summary["Encounters"].sum() == n_encounters

frequency_summary.loc[len(frequency_summary)] = [
    "Total",
    frequency_summary["Distinct Strings"].sum(),
    frequency_summary["Encounters"].sum(),
    frequency_summary["Distinct Strings (%)"].sum(),
    frequency_summary["Encounters (%)"].sum()
]

print("Complaint Frequency Distribution")
display(frequency_summary.round(2))

# 3. Concentration of frequent complaint strings
top_n_values = [10, 50, 100, 500, 1000]

concentration_summary = pd.DataFrame({
    "Top N Complaint Strings": top_n_values,
    "Encounters Covered": [
        complaint_counts.head(n).sum()
        for n in top_n_values
    ]
})

concentration_summary["Coverage (%)"] = (
    concentration_summary["Encounters Covered"]
    / n_encounters * 100
)

print("Concentration of Frequent Complaint Strings")
display(concentration_summary.round(2))

Text Diversity Summary


,Metric,Value
0,Total available complaints,425065.00
1,Distinct complaint strings,60407.00
2,Strings appearing once,44669.00
3,Strings appearing more than once,15738.00
4,Encounters with repeated strings,380396.00
5,Distinct-to-total ratio (%),14.21
6,Singleton strings (%),73.95
7,Encounters with repeated strings (%),89.49


Complaint Frequency Distribution


,Frequency Group,Distinct Strings,Encounters,Distinct Strings (%),Encounters (%)
0,1,44669,44669,73.95,10.51
1,2–5,10549,28922,17.46,6.80
2,6–10,2059,15587,3.41,3.67
3,11–100,2607,75871,4.32,17.85
4,101–1000,471,128364,0.78,30.20
5,1001+,52,131652,0.09,30.97
6,Total,60407,425065,100.00,100.00


Concentration of Frequent Complaint Strings


,Top N Complaint Strings,Encounters Covered,Coverage (%)
0,10,59105,13.90
1,50,129639,30.50
2,100,167493,39.40
3,500,257637,60.61
4,1000,291495,68.58


**Observation:** The 425,065 available chief complaints contain 60,407 distinct text strings. Although 73.95% of distinct strings appear only once, 89.49% of encounters contain complaint strings that occur more than once.

The frequency distribution exhibits a clear long-tail pattern. Only 52 distinct complaint strings occur more than 1,000 times, accounting for 30.97% of all encounters. The 100 most frequent strings cover 39.40% of encounters, while the 1,000 most frequent strings cover 68.58%.

Together with the short text lengths observed earlier, these findings indicate that the dataset contains both highly repetitive clinical expressions and a substantial number of infrequent expressions. This provides relevant context for comparing lexical and contextual NLP representations and examining their training data efficiency.

### Text Formatting and Quality

Beyond text length and repetition, chief complaints may vary in capitalization, punctuation, numeric content and whitespace usage. These characteristics are particularly relevant to clinical texts, where symbols and formatting may convey information such as symptom duration, severity, abbreviations and uncertainty.

I use Unicode character categories to distinguish letters, numbers, punctuation, symbols and other characters. I also examine whitespace patterns and unusual text formats to identify potential data quality concerns.

The analysis is descriptive and does not assume that non-alphanumeric characters or unusual formatting are necessarily errors. The results will inform the common preprocessing decisions in Section 3.3.

In [4]:
import unicodedata
from collections import Counter

# Use available raw chief complaints from previous analysis
n_encounters = len(text)

def summarize_masks(masks, category_name, add_total=False):
    """Summarize encounter counts and percentages."""
    summary = pd.DataFrame({
        category_name: list(masks.keys()),
        "Encounters": [mask.sum() for mask in masks.values()]
    })

    summary["Percentage (%)"] = (
        summary["Encounters"] / n_encounters * 100
    )

    if add_total:
        assert summary["Encounters"].sum() == n_encounters

        summary.loc[len(summary)] = [
            "Total",
            summary["Encounters"].sum(),
            summary["Percentage (%)"].sum()
        ]

    return summary.round(2)


# 1. Capitalization patterns
has_cased_letters = text.map(
    lambda s: s.lower() != s.upper()
)

lowercase_mask = text.str.islower()
uppercase_mask = text.str.isupper()

capitalization_masks = {
    "All lowercase": lowercase_mask,
    "All uppercase": uppercase_mask,
    "Mixed case": (
        has_cased_letters
        & ~lowercase_mask
        & ~uppercase_mask
    ),
    "No cased letters": ~has_cased_letters
}

print("Capitalization Patterns")
display(
    summarize_masks(
        capitalization_masks,
        "Capitalization",
        add_total=True
    )
)


# 2. Character composition and unusual text formats
def get_character_types(s):
    """Identify Unicode character types present in a string."""
    return {
        "Whitespace" if c.isspace()
        else unicodedata.category(c)[0]
        for c in s
    }

character_types = text.map(get_character_types)

composition_masks = {
    "Contains letters": character_types.map(lambda x: "L" in x),
    "Contains numbers": character_types.map(lambda x: "N" in x),
    "Contains punctuation": character_types.map(lambda x: "P" in x),
    "Contains symbols": character_types.map(lambda x: "S" in x),
    "Contains whitespace": character_types.map(lambda x: "Whitespace" in x),
    "Contains other characters": character_types.map(
        lambda x: bool(x - {"L", "N", "P", "S", "Whitespace"})
    ),
    "Numbers only (ignoring whitespace)": character_types.map(
        lambda x: x - {"Whitespace"} == {"N"}
    ),
    "Punctuation/symbols only (ignoring whitespace)": character_types.map(
        lambda x: bool(x - {"Whitespace"})
        and (x - {"Whitespace"}).issubset({"P", "S"})
    ),
    "No letters or numbers": character_types.map(
        lambda x: not bool(x & {"L", "N"})
    )
}

print("Character Composition and Unusual Text Formats")
display(
    summarize_masks(composition_masks, "Characteristic")
)


# 3. Punctuation and symbol frequency
symbol_counts = Counter()
symbol_encounter_counts = Counter()

for complaint in text:
    characters = [
        c for c in complaint
        if unicodedata.category(c)[0] in {"P", "S"}
    ]

    symbol_counts.update(characters)
    symbol_encounter_counts.update(set(characters))

symbol_summary = pd.DataFrame([
    {
        "Character": c,
        "Unicode Name": unicodedata.name(c, "Unknown"),
        "Category": (
            "Punctuation"
            if unicodedata.category(c).startswith("P")
            else "Symbol"
        ),
        "Total Occurrences": count,
        "Encounters": symbol_encounter_counts[c],
        "Encounters (%)": (
            symbol_encounter_counts[c] / n_encounters * 100
        )
    }
    for c, count in symbol_counts.items()
])

symbol_summary = symbol_summary.sort_values(
    "Encounters", ascending=False
).reset_index(drop=True)

print("Top 20 Punctuation and Symbols")
display(symbol_summary.head(20).round(2))

print(f"Distinct punctuation and symbols: {len(symbol_summary):,}")


# 4. Whitespace characteristics
whitespace_masks = {
    "Leading whitespace": text.str.contains(r"^\s", regex=True),
    "Trailing whitespace": text.str.contains(r"\s$", regex=True),
    "Consecutive whitespace": text.str.contains(r"\s{2,}", regex=True),
    "Tabs": text.str.contains("\t", regex=False),
    "Newlines": text.str.contains(r"[\r\n]", regex=True),
    "Non-standard whitespace": text.str.contains(r"[^\S ]", regex=True),
    "Affected by whitespace normalization": (
        text.str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .ne(text)
    )
}

print("Whitespace Characteristics")
display(
    summarize_masks(whitespace_masks, "Characteristic")
)

Capitalization Patterns


,Capitalization,Encounters,Percentage (%)
0,All lowercase,2016,0.47
1,All uppercase,138263,32.53
2,Mixed case,283477,66.69
3,No cased letters,1309,0.31
4,Total,425065,100.00


Character Composition and Unusual Text Formats


,Characteristic,Encounters,Percentage (%)
0,Contains letters,423757,99.69
1,Contains numbers,1021,0.24
2,Contains punctuation,171360,40.31
3,Contains symbols,445,0.10
4,Contains whitespace,323079,76.01
5,Contains other characters,0,0.00
6,Numbers only (ignoring whitespace),7,0.00
7,Punctuation/symbols only (ignoring whitespace),1301,0.31
8,No letters or numbers,1301,0.31


Top 20 Punctuation and Symbols


,Character,Unicode Name,Category,Total Occurrences,Encounters,Encounters (%)
0,",",COMMA,Punctuation,160694,138378,32.55
1,/,SOLIDUS,Punctuation,56461,50791,11.95
2,-,HYPHEN-MINUS,Punctuation,2028,2019,0.47
3,_,LOW LINE,Punctuation,6100,2005,0.47
4,?,QUESTION MARK,Punctuation,1373,1354,0.32
5,),RIGHT PARENTHESIS,Punctuation,1132,1132,0.27
6,(,LEFT PARENTHESIS,Punctuation,1132,1132,0.27
7,.,FULL STOP,Punctuation,638,543,0.13
8,+,PLUS SIGN,Symbol,423,416,0.10
9,',APOSTROPHE,Punctuation,290,265,0.06


Distinct punctuation and symbols: 27
Whitespace Characteristics


,Characteristic,Encounters,Percentage (%)
0,Leading whitespace,45,0.01
1,Trailing whitespace,1540,0.36
2,Consecutive whitespace,1240,0.29
3,Tabs,0,0.00
4,Newlines,0,0.00
5,Non-standard whitespace,0,0.00
6,Affected by whitespace normalization,2774,0.65


In [5]:
# Examine unusual text formats
unusual_masks = {
    "Numbers only": composition_masks[
        "Numbers only (ignoring whitespace)"
    ],
    "Punctuation/symbols only": composition_masks[
        "Punctuation/symbols only (ignoring whitespace)"
    ]
}

for category, mask in unusual_masks.items():
    unusual_counts = (
        text.loc[mask]
        .value_counts()
        .rename_axis("Chief Complaint")
        .reset_index(name="Encounters")
    )

    unusual_counts["Percentage within Category (%)"] = (
        unusual_counts["Encounters"] / mask.sum() * 100
    )

    print(f"{category}: {mask.sum():,} encounters")
    print(f"Distinct strings: {len(unusual_counts):,}")

    # Avoid displaying raw numeric strings
    if category == "Numbers only":
        display(
            text.loc[mask].str.strip().str.len()
            .value_counts()
            .sort_index()
            .rename_axis("Character Count")
            .reset_index(name="Encounters")
        )
    else:
        display(unusual_counts.round(2))

Numbers only: 7 encounters
Distinct strings: 6


,Character Count,Encounters
0,1,4
1,2,1
2,3,2


Punctuation/symbols only: 1,301 encounters
Distinct strings: 15


,Chief Complaint,Encounters,Percentage within Category (%)
0,___,1252,96.23
1,-,15,1.15
2,\,7,0.54
3,___ ___,6,0.46
4,.,5,0.38
5,"___, ___",5,0.38
6,?,2,0.15
7,? ___,2,0.15
8,€‡,1,0.08
9,',1,0.08


**Observation:** Capitalization varies considerably across chief complaints, with 32.53% recorded entirely in uppercase and 66.69% in mixed case. Punctuation is present in 40.31% of complaints, with commas and slashes being the most frequent characters. These findings support preserving capitalization and punctuation during common preprocessing, given their potential linguistic and clinical relevance.

Whitespace irregularities are relatively uncommon, with only 2,774 complaints (0.65%) affected by whitespace normalization.

The quality assessment identified 1,301 complaints consisting entirely of punctuation or symbols, of which 1,252 contain only three underscores. These strings appear to represent placeholders or otherwise non-informative text. I therefore exclude punctuation- and symbol-only complaints during common preprocessing while retaining numeric-only complaints and valid short texts.

## 3.3 Common Text Preprocessing

### Literature-Informed Preprocessing Strategy

The chief complaint analysis in Section 3.2 shows that the available texts are extremely short, highly repetitive and variable in formatting. Given these characteristics, text preprocessing requires careful consideration, as removing or modifying even a single word or symbol may affect the information available to an NLP model.

Previous studies have adopted different preprocessing strategies depending on their text sources, prediction tasks and modeling approaches. I reviewed both the reported methodologies and, where available, the implementation code to inform the preprocessing strategy for this study.

| Study | Text Source | Preprocessing Methods | Implications for This Study |
|---|---|---|---|
| Lee et al. (2019) | ED chief complaints | Lowercasing, special character removal and conversion of digits to written words. Abbreviation expansion and spelling correction were not applied. | Provides a reference for conventional chief complaint preprocessing without extensive clinical normalization. |
| Valmianski et al. (2019) | Patient-generated short text | Lowercasing, stopword removal, number replacement and tokenization for TF-IDF. | Demonstrates that preprocessing can be specific to the representation method rather than shared across all models. |
| Chang et al. (2020) | ED chief complaints | Exclusion of records with missing required inputs, BERT-based tokenization and sequence truncation. | Supports using model-specific input preparation for contextual language models. |
| Miotto et al. (2020) | Primary care clinical notes | Tokenization, sentence segmentation, punctuation and number removal, lemmatization, spelling correction and short-document filtering. | Provides relevant LBP-specific NLP methods, although preprocessing designed for longer clinical notes may not be appropriate for extremely short complaints. |
| Xie et al. (2022) | MIMIC-IV-ED records | General data quality checks, encounter-level data integration and controlled experimental preparation. | Provides methodological guidance for dataset integrity and consistent model evaluation rather than text-specific preprocessing. |
| Weidman et al. (2026) | EMS chief complaints | Stopword removal, lemmatization and vocabulary filtering. Source code additionally confirms lowercasing, punctuation replacement, whitespace normalization, regex tokenization and within-complaint token deduplication. | Demonstrates the feasibility of lexical modeling on extremely short clinical texts. My study adopts a more conservative common preprocessing strategy and restricts corpus-dependent transformations to the current training subset. |

These studies do not establish a universally optimal preprocessing strategy. Their methods reflect different research objectives, text characteristics and model requirements.

Since this study compares rule-based, lexical, contextual and large language model approaches, I distinguish common preprocessing from model-specific preprocessing. Common preprocessing establishes a consistent set of eligible inputs while preserving potentially useful textual information. Additional transformations are handled within individual model pipelines.

### Common Preprocessing Decisions

Based on the literature review and the observed characteristics of the chief complaints in Section 3.2, I adopt a minimal common preprocessing strategy that separates text eligibility screening from text normalization.

| Category | Operation | Decision | Rationale |
|---|---|---|---|
| Text eligibility | Missing, empty or whitespace-only complaints | Exclude | These records do not provide available text for prediction. |
| Text eligibility | Punctuation- or symbol-only complaints | Exclude | The quality assessment identified 1,301 such complaints, predominantly underscore placeholders. |
| Text eligibility | Numeric-only complaints | Retain | Only seven records were identified, with insufficient evidence to justify further exclusion based solely on their format. |
| Text normalization | Leading and trailing whitespace | Remove | Eliminates unnecessary formatting differences without modifying non-whitespace characters. |
| Text normalization | Consecutive whitespace | Normalize | Provides consistent spacing. Only 0.65% of available complaints are affected by the combined whitespace normalization operations. |
| Information preservation | Capitalization | Preserve | A substantial proportion of complaints are recorded in uppercase or mixed case. Case normalization is left to individual model pipelines. |
| Information preservation | Punctuation and symbols | Preserve | Punctuation occurs in 40.31% of complaints and may convey information through abbreviations, symptom separation and clinical notation. |
| Information preservation | Numbers | Preserve | Numbers may describe symptom duration, severity or other clinical information. |
| Information preservation | Stopwords and negation | Preserve | Removing common words may alter contextual relationships or eliminate negation. |
| Information preservation | Abbreviations | Preserve | Abbreviations may have multiple interpretations, making automatic expansion potentially unreliable. |
| Information preservation | Spelling variations | Preserve | Automatic correction may introduce unintended changes to clinical terminology. |
| Information preservation | Stemming and lemmatization | Do not apply | Word-form normalization is not required by all NLP representations. |
| Information preservation | Within-complaint token deduplication | Do not apply | Repeated words may carry information about the original expression. |
| Information preservation | Minimum word count filtering | Do not apply | Approximately 24.09% of available complaints contain only one word. |
| Encounter preservation | Identical complaint strings | Retain | Identical text from different encounters does not constitute duplicate encounter records. |
| Semantic normalization | Medical ontology mapping | Do not apply | Semantic mapping introduces additional assumptions and is outside the scope of common preprocessing. |

The objective is not to determine the optimal preprocessing configuration for every NLP model, but to establish a consistent and information-preserving input dataset for the comparative experiments.

Common preprocessing consists of deterministic, record-level operations that do not require ICD reference labels or corpus-level statistics. These operations are therefore applied before dataset splitting.

Corpus-dependent transformations, including vocabulary construction and IDF estimation, are fitted only on the current training subset. Model-specific tokenization, case normalization, sequence truncation and prompt formatting are handled during model development.

### Preprocessing Implementation

Following the preprocessing decisions established above, I first exclude complaints consisting entirely of punctuation or symbols from the available text records identified in Section 3.2.

I then normalize whitespace in the remaining complaints while preserving all other characters. The original chief complaint is retained alongside the cleaned text to support subsequent impact assessment and validation.

No ICD reference labels or corpus-dependent transformations are used at this stage.

In [6]:
# 1. Identify punctuation/symbol-only complaints
symbol_only_mask = composition_masks[
    "Punctuation/symbols only (ignoring whitespace)"
]

# Ensure the exclusion mask aligns with the available data
assert symbol_only_mask.index.equals(chief_available.index)

# 2. Retain eligible chief complaints
chief_clean = chief_available.loc[~symbol_only_mask].copy()

# 3. Apply common whitespace normalization
def normalize_whitespace(series):
    return (
        series.str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

chief_clean["chiefcomplaint_clean"] = normalize_whitespace(
    chief_clean["chiefcomplaint"]
)

# 4. Track text modifications
modified_mask = chief_clean["chiefcomplaint"].ne(
    chief_clean["chiefcomplaint_clean"]
)

# Implementation summary
print(f"Raw complaint records: {len(chief_raw):,}")
print(f"Available complaints: {len(chief_available):,}")
print(f"Punctuation/symbol-only excluded: {symbol_only_mask.sum():,}")
print(f"Eligible complaints: {len(chief_clean):,}")
print(f"Complaints modified by normalization: {modified_mask.sum():,}")
print("Common preprocessing completed.")

Raw complaint records: 425,087
Available complaints: 425,065
Punctuation/symbol-only excluded: 1,301
Eligible complaints: 423,764
Complaints modified by normalization: 2,773
Common preprocessing completed.


### Preprocessing Impact and Validation

I evaluate the impact of common preprocessing at two levels: encounter eligibility and text normalization.

The eligibility assessment accounts for all excluded and retained records. The before-and-after comparison uses the same set of eligible encounters to isolate the effects of whitespace normalization from those of record exclusion.

Finally, I validate encounter integrity, text eligibility, information preservation and preprocessing consistency to ensure that the implemented operations follow the established preprocessing decisions.

In [7]:
# 1. Text eligibility summary
eligibility_summary = pd.DataFrame({
    "Status": [
        "Missing or empty",
        "Whitespace only",
        "Punctuation/symbols only",
        "Retained eligible complaints"
    ],
    "Encounters": [
        missing_mask.sum() + empty_mask.sum(),
        whitespace_mask.sum(),
        symbol_only_mask.sum(),
        len(chief_clean)
    ]
})

eligibility_summary["Percentage (%)"] = (
    eligibility_summary["Encounters"] / len(chief_raw) * 100
)

assert eligibility_summary["Encounters"].sum() == len(chief_raw)

eligibility_summary.loc[len(eligibility_summary)] = [
    "Total",
    eligibility_summary["Encounters"].sum(),
    eligibility_summary["Percentage (%)"].sum()
]

print("Text Eligibility Summary")
display(eligibility_summary.round(4))


# 2. Before-and-after text normalization
before = chief_clean["chiefcomplaint"]
after = chief_clean["chiefcomplaint_clean"]

def describe_text(series):
    return {
        "Encounters": len(series),
        "Distinct complaint strings": series.nunique(),
        "Mean character count": series.str.len().mean(),
        "Median character count": series.str.len().median(),
        "Mean word count": series.str.split().str.len().mean(),
        "Median word count": series.str.split().str.len().median(),
        "Leading/trailing whitespace": (
            series.str.contains(r"^\s|\s$", regex=True).sum()
        ),
        "Consecutive whitespace": (
            series.str.contains(r"\s{2,}", regex=True).sum()
        )
    }

impact_summary = pd.DataFrame({
    "Before": describe_text(before),
    "After": describe_text(after)
})

impact_summary["Change"] = (
    impact_summary["After"] - impact_summary["Before"]
)

print("Before and After Preprocessing")
display(impact_summary.round(2))


# Normalization impact
normalization_summary = pd.DataFrame({
    "Status": ["Modified", "Unchanged"],
    "Encounters": [
        modified_mask.sum(),
        (~modified_mask).sum()
    ]
})

normalization_summary["Percentage (%)"] = (
    normalization_summary["Encounters"] / len(chief_clean) * 100
)

normalization_summary.loc[len(normalization_summary)] = [
    "Total",
    normalization_summary["Encounters"].sum(),
    normalization_summary["Percentage (%)"].sum()
]

print("Whitespace Normalization Impact")
display(normalization_summary.round(4))


# 3. Preprocessing validation
clean_character_types = after.map(get_character_types)

clean_symbol_only = clean_character_types.map(
    lambda x: bool(x - {"Whitespace"})
    and (x - {"Whitespace"}).issubset({"P", "S"})
)

numeric_only_before = composition_masks[
    "Numbers only (ignoring whitespace)"
]

numeric_only_after = clean_character_types.map(
    lambda x: x - {"Whitespace"} == {"N"}
)

expected_eligible = chief_available.loc[~symbol_only_mask]

validation_checks = {
    "Expected eligible encounter count": (
        len(chief_clean) == 423764
    ),
    "No missing or empty stay_id": (
        chief_clean["stay_id"].notna().all()
        and chief_clean["stay_id"].astype(str).str.strip().ne("").all()
    ),
    "Unique stay_id": (
        chief_clean["stay_id"].is_unique
    ),
    "Eligible encounter identifiers preserved": (
        chief_clean["stay_id"].equals(expected_eligible["stay_id"])
    ),
    "No missing or blank cleaned text": (
        after.notna().all()
        and after.str.strip().ne("").all()
    ),
    "No punctuation/symbol-only complaints": (
        not clean_symbol_only.any()
    ),
    "Numeric-only encounters retained": (
        after.index[numeric_only_after].equals(
            chief_available.index[numeric_only_before]
        )
    ),
    "No leading or trailing whitespace": (
        not after.str.contains(r"^\s|\s$", regex=True).any()
    ),
    "No consecutive whitespace": (
        not after.str.contains(r"\s{2,}", regex=True).any()
    ),
    "Non-whitespace characters preserved": (
        before.str.replace(r"\s+", "", regex=True).equals(
            after.str.replace(r"\s+", "", regex=True)
        )
    ),
    "Preprocessing is idempotent": (
        normalize_whitespace(after).equals(after)
    ),
    "Original text preserved in cleaned dataset": (
        before.equals(
            chief_raw.loc[chief_clean.index, "chiefcomplaint"]
        )
    )
}

validation_summary = pd.DataFrame({
    "Check": list(validation_checks.keys()),
    "Passed": list(validation_checks.values())
})

print("Preprocessing Validation")
display(validation_summary)

assert validation_summary["Passed"].all(), (
    "Preprocessing validation failed"
)

print("Preprocessing validation: PASSED")

Text Eligibility Summary


,Status,Encounters,Percentage (%)
0,Missing or empty,22,0.0052
1,Whitespace only,0,0.0000
2,Punctuation/symbols only,1301,0.3061
3,Retained eligible complaints,423764,99.6888
4,Total,425087,100.0000


Before and After Preprocessing


,Before,After,Change
Encounters,423764.00,423764.00,0.00
Distinct complaint strings,60392.00,59432.00,-960.00
Mean character count,14.59,14.58,-0.01
Median character count,13.00,13.00,0.00
Mean word count,2.43,2.43,0.00
Median word count,2.00,2.00,0.00
Leading/trailing whitespace,1580.00,0.00,-1580.00
Consecutive whitespace,1240.00,0.00,-1240.00


Whitespace Normalization Impact


,Status,Encounters,Percentage (%)
0,Modified,2773,0.6544
1,Unchanged,420991,99.3456
2,Total,423764,100.0000


Preprocessing Validation


,Check,Passed
0,Expected eligible encounter count,True
1,No missing or empty stay_id,True
2,Unique stay_id,True
3,Eligible encounter identifiers preserved,True
4,No missing or blank cleaned text,True
5,No punctuation/symbol-only complaints,True
6,Numeric-only encounters retained,True
7,No leading or trailing whitespace,True
8,No consecutive whitespace,True
9,Non-whitespace characters preserved,True


Preprocessing validation: PASSED


**Observation:** Common preprocessing retained 423,764 of the original 425,087 chief complaint records (99.69%). A total of 1,323 records were excluded, including 22 missing or empty complaints and 1,301 punctuation- or symbol-only complaints.

Whitespace normalization modified only 2,773 eligible complaints (0.65%) but reduced the number of distinct complaint strings from 60,392 to 59,432. This indicates that a portion of the observed text diversity was attributable to formatting differences rather than differences in non-whitespace content. Encounter counts and word counts remained unchanged during normalization.

All 12 validation checks passed, confirming encounter integrity, text eligibility, preservation of non-whitespace characters and preprocessing consistency. The resulting dataset provides a common text input for subsequent model-specific processing.

## 3.4 Encounter-Level Dataset Integration

With common text preprocessing completed, I integrate the eligible chief complaints with encounter identifiers and the ICD-based reference labels constructed in P2.

The integration is performed at the encounter level using `stay_id` as the join key. Patient identifiers are retained to support patient-grouped data splitting in subsequent sections.

Only encounters with eligible chief complaint text and an available ICD-based reference label are included in the modeling dataset. Encounters without documented diagnoses are not treated as negative cases.

### Input Data and Validation

Three datasets are required for integration:

| Data Source | Dataset | Key Information |
|---|---|---|
| P1 — Dataset Exploration | `edstays_raw.csv` | Encounter and patient identifiers |
| P2 — Reference Label Construction | `encounter_reference_labels.csv` | Encounter-level binary LBP reference labels |
| P3 — Common Text Preprocessing | `chief_clean` | Original and preprocessed chief complaints |

Before integration, I verify the expected structure, record counts and uniqueness of encounter identifiers in each dataset.

In [8]:
# 1. Load encounter identifiers and ICD reference labels
edstays = pd.read_csv(
    INPUT_DIR / "edstays_raw.csv",
    keep_default_na=False
)

reference_labels = pd.read_csv(
    LABEL_DIR / "encounter_reference_labels.csv",
    keep_default_na=False
)

# chief_clean is retained in memory from Section 3.3

# 2. Input dataset inventory
input_inventory = pd.DataFrame({
    "Dataset": [
        "Encounter identifiers",
        "Eligible chief complaints",
        "ICD reference labels"
    ],
    "Records": [
        len(edstays),
        len(chief_clean),
        len(reference_labels)
    ],
    "Unique stay_id": [
        edstays["stay_id"].nunique(),
        chief_clean["stay_id"].nunique(),
        reference_labels["stay_id"].nunique()
    ],
    "Columns": [
        edstays.shape[1],
        chief_clean.shape[1],
        reference_labels.shape[1]
    ]
})

print("Input Dataset Inventory")
display(input_inventory)


# 3. Validate input datasets
datasets = {
    "Encounter identifiers": {
        "data": edstays,
        "expected_rows": 425087,
        "expected_columns": {"stay_id", "subject_id"}
    },
    "Eligible chief complaints": {
        "data": chief_clean,
        "expected_rows": 423764,
        "expected_columns": {
            "stay_id",
            "chiefcomplaint",
            "chiefcomplaint_clean"
        }
    },
    "ICD reference labels": {
        "data": reference_labels,
        "expected_rows": 423989,
        "expected_columns": {"stay_id", "lbp_label"}
    }
}

validation_records = []

for name, config in datasets.items():
    df = config["data"]

    validation_records.append({
        "Dataset": name,
        "Expected Columns": (
            set(df.columns) == config["expected_columns"]
        ),
        "Expected Rows": (
            len(df) == config["expected_rows"]
        ),
        "Complete stay_id": (
            df["stay_id"].notna().all()
            and df["stay_id"].astype(str).str.strip().ne("").all()
        ),
        "Unique stay_id": df["stay_id"].is_unique
    })

input_validation = pd.DataFrame(validation_records)

print("Input Dataset Validation")
display(input_validation)

assert input_validation.drop(columns="Dataset").all().all()


# 4. Validate integration prerequisites
integration_checks = {
    "Complete patient identifiers": (
        edstays["subject_id"].notna().all()
        and edstays["subject_id"].astype(str).str.strip().ne("").all()
    ),
    "Valid binary reference labels": (
        reference_labels["lbp_label"].isin([0, 1]).all()
    ),
    "All chief complaint encounters exist in edstays": (
        chief_clean["stay_id"].isin(edstays["stay_id"]).all()
    ),
    "All reference label encounters exist in edstays": (
        reference_labels["stay_id"].isin(edstays["stay_id"]).all()
    ),
    "Consistent stay_id data types": (
        edstays["stay_id"].dtype
        == chief_clean["stay_id"].dtype
        == reference_labels["stay_id"].dtype
    )
}

integration_validation = pd.DataFrame({
    "Check": list(integration_checks.keys()),
    "Passed": list(integration_checks.values())
})

print("Integration Prerequisites")
display(integration_validation)

assert integration_validation["Passed"].all()

print("Input dataset validation: PASSED")

Input Dataset Inventory


,Dataset,Records,Unique stay_id,Columns
0,Encounter identifiers,425087,425087,2
1,Eligible chief complaints,423764,423764,3
2,ICD reference labels,423989,423989,2


Input Dataset Validation


,Dataset,Expected Columns,Expected Rows,Complete stay_id,Unique stay_id
0,Encounter identifiers,True,True,True,True
1,Eligible chief complaints,True,True,True,True
2,ICD reference labels,True,True,True,True


Integration Prerequisites


,Check,Passed
0,Complete patient identifiers,True
1,Valid binary reference labels,True
2,All chief complaint encounters exist in edstays,True
3,All reference label encounters exist in edstays,True
4,Consistent stay_id data types,True


Input dataset validation: PASSED


### Encounter-Level Integration

The modeling dataset requires both eligible chief complaint text and an available ICD-based reference label.

I first examine the overlap between text eligibility and reference label availability across all ED encounters. This accounts for encounters that may fail one or both eligibility conditions and provides a complete record of cohort formation.

The eligible chief complaints, encounter identifiers and reference labels are then integrated using one-to-one joins on `stay_id`. Each resulting row represents one ED encounter, with `subject_id` retained for patient-grouped data splitting.

In [9]:
# 1. Examine overlap between text and label eligibility
encounter_eligibility = edstays[["stay_id"]].copy()

encounter_eligibility["eligible_text"] = (
    encounter_eligibility["stay_id"].isin(chief_clean["stay_id"])
)

encounter_eligibility["available_label"] = (
    encounter_eligibility["stay_id"].isin(reference_labels["stay_id"])
)

has_text = encounter_eligibility["eligible_text"]
has_label = encounter_eligibility["available_label"]

eligibility_categories = {
    "Eligible text and available label": has_text & has_label,
    "Eligible text only": has_text & ~has_label,
    "Available label only": ~has_text & has_label,
    "Neither eligible text nor available label": ~has_text & ~has_label
}

overlap_summary = pd.DataFrame({
    "Eligibility Status": list(eligibility_categories.keys()),
    "Encounters": [
        mask.sum() for mask in eligibility_categories.values()
    ]
})

overlap_summary["Percentage (%)"] = (
    overlap_summary["Encounters"] / len(edstays) * 100
)

assert overlap_summary["Encounters"].sum() == len(edstays)

overlap_summary.loc[len(overlap_summary)] = [
    "Total",
    overlap_summary["Encounters"].sum(),
    overlap_summary["Percentage (%)"].sum()
]

print("Text and Reference Label Eligibility")
display(overlap_summary.round(4))


# 2. Integrate eligible encounters
modeling_dataset = (
    edstays
    .merge(
        chief_clean,
        on="stay_id",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        reference_labels,
        on="stay_id",
        how="inner",
        validate="one_to_one"
    )
)

# Arrange columns consistently
modeling_dataset = modeling_dataset[
    [
        "stay_id",
        "subject_id",
        "chiefcomplaint",
        "chiefcomplaint_clean",
        "lbp_label"
    ]
]


# 3. Summarize cohort formation
n_raw = len(edstays)
n_text_eligible = len(chief_clean)
n_final = len(modeling_dataset)

integration_summary = pd.DataFrame({
    "Stage": [
        "Original ED encounters",
        "After text eligibility screening",
        "After reference label integration"
    ],
    "Remaining Encounters": [
        n_raw,
        n_text_eligible,
        n_final
    ],
    "Excluded at Stage": [
        0,
        n_raw - n_text_eligible,
        n_text_eligible - n_final
    ]
})

integration_summary["Retained (%)"] = (
    integration_summary["Remaining Encounters"] / n_raw * 100
)

print("Cohort Formation Summary")
display(integration_summary.round(4))


# 4. Examine integrated dataset structure
dataset_summary = pd.DataFrame({
    "Metric": [
        "Total encounters",
        "Unique encounters",
        "Unique patients",
        "Total columns",
        "Total excluded encounters"
    ],
    "Value": [
        len(modeling_dataset),
        modeling_dataset["stay_id"].nunique(),
        modeling_dataset["subject_id"].nunique(),
        modeling_dataset.shape[1],
        n_raw - n_final
    ]
})

print("Integrated Dataset Summary")
display(dataset_summary)

column_summary = pd.DataFrame({
    "Column": modeling_dataset.columns,
    "Data Type": modeling_dataset.dtypes.astype(str).values
})

print("Integrated Dataset Structure")
display(column_summary)


# 5. Verify integration results
expected_ids = encounter_eligibility.loc[
    has_text & has_label, "stay_id"
]

assert len(modeling_dataset) == len(expected_ids)
assert modeling_dataset["stay_id"].is_unique

assert set(modeling_dataset["stay_id"]) == set(expected_ids)

assert integration_summary["Excluded at Stage"].sum() == (
    n_raw - n_final
)

print("Encounter-level integration: PASSED")

Text and Reference Label Eligibility


,Eligibility Status,Encounters,Percentage (%)
0,Eligible text and available label,422682,99.4342
1,Eligible text only,1082,0.2545
2,Available label only,1307,0.3075
3,Neither eligible text nor available label,16,0.0038
4,Total,425087,100.0000


Cohort Formation Summary


,Stage,Remaining Encounters,Excluded at Stage,Retained (%)
0,Original ED encounters,425087,0,100.0000
1,After text eligibility screening,423764,1323,99.6888
2,After reference label integration,422682,1082,99.4342


Integrated Dataset Summary


,Metric,Value
0,Total encounters,422682
1,Unique encounters,422682
2,Unique patients,204509
3,Total columns,5
4,Total excluded encounters,2405


Integrated Dataset Structure


,Column,Data Type
0,stay_id,int64
1,subject_id,int64
2,chiefcomplaint,str
3,chiefcomplaint_clean,str
4,lbp_label,int64


Encounter-level integration: PASSED


**Observation:** The encounter-level integration retained 422,682 ED encounters from 204,509 unique patients, representing 99.43% of the original ED encounters.

A total of 2,405 encounters were excluded through text eligibility screening and reference label availability requirements. Sixteen encounters failed both eligibility conditions and were accounted for only once.

The integrated dataset contains one row per encounter, with encounter and patient identifiers, original and preprocessed chief complaints, and the binary ICD-based LBP reference label.

### Modeling Dataset Export

The integrated encounter-level dataset is saved for subsequent experimental preparation and model development. This dataset contains eligible chief complaints, encounter and patient identifiers, and ICD-based binary reference labels before experimental splitting.

In [10]:
# Export the integrated modeling dataset
modeling_path = RESULTS_DIR / "modeling_dataset.csv"

modeling_dataset.to_csv(modeling_path, index=False)

print(f"Dataset saved to: {modeling_path}")
print(f"Encounters: {len(modeling_dataset):,}")
print(f"Columns: {modeling_dataset.shape[1]}")

Dataset saved to: ../results/03_experimental_setup/modeling_dataset.csv
Encounters: 422,682
Columns: 5


## 3.5 Dataset Validation and Class Distribution

With the encounter-level modeling dataset constructed and saved, I verify the exported dataset against the integrated DataFrame and examine the final reference label distribution.

The class distribution provides the basis for subsequent experimental splitting, model development and performance evaluation.

In [11]:
# 1. Verify the exported modeling dataset
saved_dataset = pd.read_csv(
    modeling_path,
    keep_default_na=False
)

assert saved_dataset.equals(modeling_dataset), (
    "Exported dataset differs from the integrated dataset"
)

print("Dataset export validation: PASSED")


# 2. Final reference label distribution
class_summary = (
    modeling_dataset["lbp_label"]
    .value_counts()
    .sort_index()
    .rename_axis("Label")
    .reset_index(name="Encounters")
)

class_summary["Reference Class"] = class_summary["Label"].map({
    0: "ICD-defined LBP negative",
    1: "ICD-defined LBP positive"
})

class_summary["Percentage (%)"] = (
    class_summary["Encounters"] / len(modeling_dataset) * 100
)

class_summary = class_summary[
    ["Reference Class", "Encounters", "Percentage (%)"]
]

class_summary.loc[len(class_summary)] = [
    "Total",
    class_summary["Encounters"].sum(),
    class_summary["Percentage (%)"].sum()
]

print("Final Reference Label Distribution")
display(class_summary.round(2))


# 3. Class imbalance
n_positive = modeling_dataset["lbp_label"].eq(1).sum()
n_negative = modeling_dataset["lbp_label"].eq(0).sum()

assert n_positive + n_negative == len(modeling_dataset)
assert n_positive > 0 and n_negative > 0

positive_prevalence = n_positive / len(modeling_dataset)
imbalance_ratio = n_negative / n_positive

print(f"Positive prevalence: {positive_prevalence:.2%}")
print(f"Negative-to-positive ratio: {imbalance_ratio:.2f}:1")

Dataset export validation: PASSED
Final Reference Label Distribution


,Reference Class,Encounters,Percentage (%)
0,ICD-defined LBP negative,406355,96.14
1,ICD-defined LBP positive,16327,3.86
2,Total,422682,100.00


Positive prevalence: 3.86%
Negative-to-positive ratio: 24.89:1


**Observation:** The final modeling dataset contains 422,682 encounters, including 16,327 ICD-defined LBP positive cases (3.86%) and 406,355 negative cases (96.14%).

Compared with the reference labels constructed in P2, text eligibility screening excluded only four positive encounters, with minimal change in overall LBP prevalence.

The negative-to-positive ratio of 24.89:1 indicates substantial class imbalance, supporting the use of positive-class F1, precision, recall and average precision rather than relying on accuracy alone.

The exported modeling dataset was successfully validated against the integrated DataFrame.

## 3.6 Patient-Grouped Data Splitting

Following the fixed benchmark principle described by Xie et al. (2022), I establish a common training, validation and test partition for all NLP models. I adapt this approach by incorporating patient-level grouping and approximate class stratification.

The unit of prediction remains the ED encounter, while the unit of splitting is the patient. Since patients may have multiple ED encounters, all encounters belonging to the same patient must remain within a single partition to prevent patient overlap between training and evaluation datasets.

Given the positive prevalence of 3.86%, maintaining approximately consistent class distributions across partitions is also important.

### Splitting Strategy

I use a two-stage StratifiedGroupKFold procedure to construct fixed training, validation and test partitions.

| Parameter | Setting |
|---|---|
| Grouping variable | `subject_id` |
| Stratification variable | `lbp_label` |
| Stage 1 | 5 folds; first fold reserved as test (20%) |
| Stage 2 | 8 folds within the development set; first fold reserved as validation (10% of the full dataset) |
| Target proportions | 70% training, 10% validation, 20% test |
| Input ordering | Sorted by `stay_id` |
| Random seed | 42 |
| Shuffle | True |

StratifiedGroupKFold is used to construct a single fixed partition rather than to perform cross-validation.

### Validation Criteria

| Criterion | Acceptance Requirement |
|---|---|
| Patient overlap | Zero |
| Encounter overlap | Zero |
| Encounter coverage | 100% |
| Partition proportion deviation | Within 0.5 percentage points of the target |
| Positive prevalence deviation | Within 0.1 percentage points of the overall prevalence |

Because patients are indivisible groups with varying encounter counts, exact partition sizes and class proportions are not guaranteed.

The training and validation partitions are used for model development and selection. The test partition remains fixed and is reserved for final evaluation after all experimental configurations have been finalized.

In [12]:
from sklearn.model_selection import StratifiedGroupKFold

# 1. Splitting configuration
RANDOM_SEED = 42
SPLIT_TOLERANCE = 0.005
PREVALENCE_TOLERANCE = 0.001

target_proportions = {
    "train": 0.70,
    "validation": 0.10,
    "test": 0.20
}

# Ensure consistent input ordering
split_data = (
    modeling_dataset
    .sort_values("stay_id")
    .reset_index(drop=True)
)

X = split_data[["stay_id"]]
y = split_data["lbp_label"]
groups = split_data["subject_id"]


# 2. Stage 1: development / test split
stage1 = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED
)

dev_idx, test_idx = next(
    stage1.split(X, y, groups)
)

development_data = split_data.iloc[dev_idx]


# 3. Stage 2: training / validation split
stage2 = StratifiedGroupKFold(
    n_splits=8,
    shuffle=True,
    random_state=RANDOM_SEED
)

train_local_idx, val_local_idx = next(
    stage2.split(
        development_data[["stay_id"]],
        development_data["lbp_label"],
        development_data["subject_id"]
    )
)

# Convert development-local positions to full dataset positions
train_idx = dev_idx[train_local_idx]
val_idx = dev_idx[val_local_idx]


# 4. Construct partition membership
partitions = {
    "train": split_data.iloc[train_idx],
    "validation": split_data.iloc[val_idx],
    "test": split_data.iloc[test_idx]
}

dataset_splits = pd.concat(
    [
        df[["stay_id"]].assign(split=name)
        for name, df in partitions.items()
    ],
    ignore_index=True
).sort_values("stay_id").reset_index(drop=True)


# 5. Summarize actual partition distributions
overall_prevalence = split_data["lbp_label"].mean()
n_total = len(split_data)

summary_records = []

for name, df in partitions.items():

    n_encounters = len(df)
    n_positive = df["lbp_label"].eq(1).sum()
    n_negative = df["lbp_label"].eq(0).sum()

    actual_proportion = n_encounters / n_total
    prevalence = n_positive / n_encounters

    summary_records.append({
        "Partition": name,
        "Encounters": n_encounters,
        "Patients": df["subject_id"].nunique(),
        "Positive": n_positive,
        "Negative": n_negative,
        "Target (%)": target_proportions[name] * 100,
        "Actual (%)": actual_proportion * 100,
        "Split Deviation (pp)": (
            actual_proportion - target_proportions[name]
        ) * 100,
        "Prevalence (%)": prevalence * 100,
        "Prevalence Deviation (pp)": (
            prevalence - overall_prevalence
        ) * 100
    })

partition_summary = pd.DataFrame(summary_records)


# 6. Validate partition integrity
patient_sets = {
    name: set(df["subject_id"])
    for name, df in partitions.items()
}

patient_overlap = any([
    bool(patient_sets["train"] & patient_sets["validation"]),
    bool(patient_sets["train"] & patient_sets["test"]),
    bool(patient_sets["validation"] & patient_sets["test"])
])

split_validation_checks = {
    "Zero patient overlap": not patient_overlap,

    "Zero encounter overlap": (
        dataset_splits["stay_id"].is_unique
    ),

    "Complete encounter coverage": (
        len(dataset_splits) == n_total
        and set(dataset_splits["stay_id"])
        == set(split_data["stay_id"])
    ),

    "Both classes present in all partitions": (
        partition_summary["Positive"].gt(0).all()
        and partition_summary["Negative"].gt(0).all()
    ),

    "Partition proportions within tolerance": (
        partition_summary["Split Deviation (pp)"]
        .abs().le(SPLIT_TOLERANCE * 100).all()
    ),

    "Positive prevalence within tolerance": (
        partition_summary["Prevalence Deviation (pp)"]
        .abs().le(PREVALENCE_TOLERANCE * 100).all()
    )
}

split_validation = pd.DataFrame({
    "Check": list(split_validation_checks.keys()),
    "Passed": list(split_validation_checks.values())
})


# 7. Add total row and display results
partition_summary.loc[len(partition_summary)] = {
    "Partition": "Total",
    "Encounters": n_total,
    "Patients": split_data["subject_id"].nunique(),
    "Positive": split_data["lbp_label"].eq(1).sum(),
    "Negative": split_data["lbp_label"].eq(0).sum(),
    "Target (%)": 100.0,
    "Actual (%)": 100.0,
    "Split Deviation (pp)": 0.0,
    "Prevalence (%)": overall_prevalence * 100,
    "Prevalence Deviation (pp)": 0.0
}

print("Experimental Partition Summary")
display(partition_summary.round(4))

print("Partition Validation")
display(split_validation)

assert split_validation["Passed"].all(), (
    "Partition validation failed. Review the reported deviations."
)

print("Patient-grouped data splitting: PASSED")

Experimental Partition Summary


,Partition,Encounters,Patients,Positive,Negative,Target (%),Actual (%),Split Deviation (pp),Prevalence (%),Prevalence Deviation (pp)
0,train,295877,143119,11429,284448,70.0,69.9999,-0.0001,3.8628,0.0000
1,validation,42268,20483,1632,40636,10.0,10.0000,-0.0000,3.8611,-0.0016
2,test,84537,40907,3266,81271,20.0,20.0001,0.0001,3.8634,0.0007
3,Total,422682,204509,16327,406355,100.0,100.0000,0.0000,3.8627,0.0000


Partition Validation


,Check,Passed
0,Zero patient overlap,True
1,Zero encounter overlap,True
2,Complete encounter coverage,True
3,Both classes present in all partitions,True
4,Partition proportions within tolerance,True
5,Positive prevalence within tolerance,True


Patient-grouped data splitting: PASSED


**Observation:** The two-stage patient-grouped splitting procedure successfully partitioned 422,682 encounters from 204,509 patients into training (295,877), validation (42,268) and test (84,537) sets.

The resulting partition proportions closely match the intended 70/10/20 split, while positive prevalence remains approximately 3.86% across all three partitions. Both deviations are well within the predefined tolerances.

All six validation checks passed, confirming complete encounter coverage, zero patient and encounter overlap, and consistent class distributions across partitions.

These partitions are fixed for all subsequent model development and evaluation.

In [13]:
# Export validated partition membership
splits_path = RESULTS_DIR / "dataset_splits.csv"

dataset_splits.to_csv(splits_path, index=False)

# Verify the exported file
saved_splits = pd.read_csv(splits_path)

assert saved_splits.equals(dataset_splits)

print(f"Dataset splits saved to: {splits_path}")
print("Dataset split export validation: PASSED")

Dataset splits saved to: ../results/03_experimental_setup/dataset_splits.csv
Dataset split export validation: PASSED


## 3.7 Training Data Subsets

Following the training data reduction approach explored by Miotto et al. (2020) and the controlled data-efficiency evaluation framework of EHRSHOT (Wornow et al., 2023), I construct nested training subsets to examine how different supervised NLP models perform under varying amounts of labeled training data.

All subsets are drawn exclusively from the training partition established in Section 3.6. Sampling is performed at the encounter level because the experimental data budget is measured by the number of labeled chief complaints. Patient separation across training, validation and test partitions is already ensured.

### Sampling Strategy

| Parameter | Setting |
|---|---|
| Sampling population | Training partition only |
| Sampling unit | Encounter (`stay_id`) |
| Sampling method | Stratified random sampling without replacement |
| Training fractions | 0.1%, 1%, 10%, 100% |
| Sampling seeds | 42, 123, 2026 |
| Nesting method | Class-specific random permutations and prefix selection |
| Rounding rule | Independent class-wise flooring |
| Minimum class requirement | Both classes must be present |
| Validation and test sets | Fixed and unchanged |
| Full-data condition | Reuse Experiment 1 results |

For each sampling seed, positive and negative encounter IDs are independently shuffled. Progressively larger prefixes are selected from each class to construct nested subsets while approximately preserving the natural class prevalence.

Class counts are fixed across sampling seeds for each training fraction, while the selected encounters may differ. All supervised models use the same subset membership for a given fraction and seed.

Learned preprocessing transformations and class weights will be fitted or calculated separately using the corresponding training subset during model development.

This experiment evaluates training data efficiency rather than strict few-shot learning or total labeling-budget efficiency, since the full labeled validation set remains available across all training conditions.

In [14]:
# 1. Sampling configuration
SAMPLING_SEEDS = [42, 123, 2026]
TRAINING_FRACTIONS = [0.001, 0.01, 0.10, 1.00]

train_data = (
    partitions["train"][["stay_id", "lbp_label"]]
    .sort_values("stay_id")
    .reset_index(drop=True)
)

positive_ids = train_data.loc[
    train_data["lbp_label"].eq(1), "stay_id"
].to_numpy()

negative_ids = train_data.loc[
    train_data["lbp_label"].eq(0), "stay_id"
].to_numpy()

n_train = len(train_data)
n_pos = len(positive_ids)
n_neg = len(negative_ids)

# Target class counts using independent class-wise flooring
target_counts = {
    fraction: {
        "positive": max(1, int(np.floor(n_pos * fraction))),
        "negative": int(np.floor(n_neg * fraction))
    }
    for fraction in TRAINING_FRACTIONS
}


# 2. Generate nested subset membership
training_subsets = train_data[["stay_id"]].copy()

summary_records = []
validation_records = []

for seed in SAMPLING_SEEDS:

    rng = np.random.default_rng(seed)

    shuffled_pos = rng.permutation(positive_ids)
    shuffled_neg = rng.permutation(negative_ids)

    # Default membership: full training set only
    membership = pd.Series(
        1.0,
        index=training_subsets["stay_id"]
    )

    subset_sets = {}

    # Assign membership from largest to smallest fraction
    for fraction in reversed(TRAINING_FRACTIONS):

        counts = target_counts[fraction]

        selected_pos = shuffled_pos[:counts["positive"]]
        selected_neg = shuffled_neg[:counts["negative"]]

        selected_ids = np.concatenate([
            selected_pos,
            selected_neg
        ])

        membership.loc[selected_ids] = fraction
        subset_sets[fraction] = set(selected_ids)

    seed_column = f"seed_{seed}"

    training_subsets[seed_column] = (
        training_subsets["stay_id"].map(membership)
    )


    # 3. Summarize and validate actual encoded membership
    for fraction in TRAINING_FRACTIONS:

        selected = training_subsets[seed_column].le(fraction)
        subset = train_data.loc[selected]

        actual_pos = subset["lbp_label"].eq(1).sum()
        actual_neg = subset["lbp_label"].eq(0).sum()
        actual_total = len(subset)

        summary_records.append({
            "Seed": seed,
            "Target Fraction (%)": fraction * 100,
            "Actual Fraction (%)": actual_total / n_train * 100,
            "Encounters": actual_total,
            "Positive": actual_pos,
            "Negative": actual_neg,
            "Prevalence (%)": actual_pos / actual_total * 100
        })

        validation_records.append({
            "Seed": seed,
            "Fraction (%)": fraction * 100,
            "Correct class counts": (
                actual_pos == target_counts[fraction]["positive"]
                and actual_neg == target_counts[fraction]["negative"]
            ),
            "Both classes present": (
                actual_pos > 0 and actual_neg > 0
            ),
            "Unique encounters": subset["stay_id"].is_unique,
            "Training encounters only": (
                subset["stay_id"].isin(train_data["stay_id"]).all()
            ),
            "Membership encoding correct": (
                set(subset["stay_id"]) == subset_sets[fraction]
            )
        })

    # Validate nesting for this seed
    assert all(
        subset_sets[smaller].issubset(subset_sets[larger])
        for smaller, larger in zip(
            TRAINING_FRACTIONS[:-1],
            TRAINING_FRACTIONS[1:]
        )
    ), f"Nested subset validation failed for seed {seed}"


# 4. Validate the complete membership file
subset_summary = pd.DataFrame(summary_records)
subset_validation = pd.DataFrame(validation_records)

assert training_subsets["stay_id"].is_unique
assert len(training_subsets) == n_train
assert not training_subsets.isna().any().any()

assert set(training_subsets["stay_id"]) == set(train_data["stay_id"])

assert training_subsets.iloc[:, 1:].isin(
    TRAINING_FRACTIONS
).all().all()

validation_columns = subset_validation.columns[2:]

assert subset_validation[validation_columns].all().all()


# 5. Display results
print("Training Subset Summary")
display(subset_summary.round(4))

print("Subset Validation Summary")
display(
    subset_validation[validation_columns]
    .all()
    .rename("Passed")
    .rename_axis("Check")
    .reset_index()
)

print("Nested subset validation: PASSED")
print("Training subset membership validation: PASSED")

Training Subset Summary


,Seed,Target Fraction (%),Actual Fraction (%),Encounters,Positive,Negative,Prevalence (%)
0,42,0.1,0.0997,295,11,284,3.7288
1,42,1.0,0.9997,2958,114,2844,3.8540
2,42,10.0,9.9994,29586,1142,28444,3.8599
3,42,100.0,100.0000,295877,11429,284448,3.8628
4,123,0.1,0.0997,295,11,284,3.7288
5,123,1.0,0.9997,2958,114,2844,3.8540
6,123,10.0,9.9994,29586,1142,28444,3.8599
7,123,100.0,100.0000,295877,11429,284448,3.8628
8,2026,0.1,0.0997,295,11,284,3.7288
9,2026,1.0,0.9997,2958,114,2844,3.8540


Subset Validation Summary


,Check,Passed
0,Correct class counts,True
1,Both classes present,True
2,Unique encounters,True
3,Training encounters only,True
4,Membership encoding correct,True


Nested subset validation: PASSED
Training subset membership validation: PASSED


In [15]:
# Export validated training subset membership
subsets_path = RESULTS_DIR / "training_subsets.csv"

training_subsets.to_csv(subsets_path, index=False)

saved_subsets = pd.read_csv(subsets_path)

assert saved_subsets.equals(training_subsets)

print(f"Training subsets saved to: {subsets_path}")
print("Training subset export validation: PASSED")

Training subsets saved to: ../results/03_experimental_setup/training_subsets.csv
Training subset export validation: PASSED


**Observation:** Nested training subsets were successfully constructed at 0.1%, 1%, 10% and 100% of the training data using three sampling seeds (42, 123 and 2026).

All validation checks passed, confirming the expected class counts, the presence of both classes, unique encounter membership, correct membership encoding and nested relationships.

The validated subset membership was exported and will be reused across supervised models to ensure consistent training data conditions in Experiment 2.

## 3.8 Experimental Dataset Summary

The dataset preparation and experimental setup are complete. The final modeling dataset contains 422,682 eligible ED encounters from 204,509 unique patients, with an ICD-defined LBP prevalence of 3.86%.

### Fixed Experimental Partitions

| Partition | Encounters | Patients | Positive | Negative |
|---|---:|---:|---:|---:|
| Training | 295,877 | 143,119 | 11,429 | 284,448 |
| Validation | 42,268 | 20,483 | 1,632 | 40,636 |
| Test | 84,537 | 40,907 | 3,266 | 81,271 |
| Total | 422,682 | 204,509 | 16,327 | 406,355 |

### Training Data Conditions

| Training Fraction | Encounters | Positive | Negative |
|---|---:|---:|---:|
| 0.1% | 295 | 11 | 284 |
| 1% | 2,958 | 114 | 2,844 |
| 10% | 29,586 | 1,142 | 28,444 |
| 100% | 295,877 | 11,429 | 284,448 |

The reduced-data subsets are nested and generated using three sampling seeds, with identical class counts across seeds. The full training condition is shared with Experiment 1.

### Output Files

| File | Rows | Columns | Purpose |
|---|---:|---:|---|
| `modeling_dataset.csv` | 422,682 | 5 | Eligible encounters, chief complaints and reference labels |
| `dataset_splits.csv` | 422,682 | 2 | Fixed training, validation and test assignments |
| `training_subsets.csv` | 295,877 | 4 | Nested training subset membership for three sampling seeds |

All three files are stored locally in `results/03_experimental_setup/` and have passed export validation.

These files establish a common experimental foundation for both the comparative NLP evaluation and the training data efficiency experiment. All subsequent models will use the fixed partitions and predefined training subsets without regenerating them.

The next stage focuses on model development, beginning with the rule-based baseline.